Дз-1 по DE

Выполнено Алгазиновым Александром

# Описание

Я загружаю весь каталог товаров из REST API DummyJSON в catalog.products в PostgreSQL. API отдает страницы по 100 товаров, а число записей мы берем из total

А еще я только что осознал что по ТЗ нужно это делать в публичном репозитории то есть это решение потенциально увидят мои хейтеры и мои фанаты

В Transform я привожу строки и типы к единому виду, раскрываю размеры товара и рассчитываю цену после скидки с округлением до двух знаков. Валюту и единицы измерения источник явно не задает, поэтому я их не придумываю

А если это потенциально увидят мои хейтеры и фанаты - мне придется воздержаться от неадекатных но смешных шуток которые я обычно пишу в решениях чтобы хоть как-то разнообразить эти скучные куски кода...

Повторный запуск целиком заменяет каталог в одной транзакции. При ошибке записи PostgreSQL откатывает и очистку, и вставку: предыдущий каталог остается на месте


In [1]:
import logging
import os
import re
import sys
from contextlib import closing
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from pathlib import Path

import pandas as pd
import psycopg2
import requests
from dotenv import find_dotenv, load_dotenv
from IPython.display import display
from psycopg2.extras import execute_values
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


## Конфигурация

Перед запуском копируем .env.example в .env, задаем пароль и поднимаем Docker Compose по инструкции из README.md. Ну а потом по классике restart kernel run all


In [2]:
env_path = Path(find_dotenv(usecwd=True, raise_error_if_not_found=True))
load_dotenv(env_path, override=False)

required_settings = [
    'POSTGRES_DB', 'POSTGRES_USER', 'POSTGRES_PASSWORD',
    'POSTGRES_HOST', 'POSTGRES_PORT', 'PRODUCTS_API_URL',
]
missing_settings = [name for name in required_settings if not os.getenv(name, '').strip()]
if missing_settings:
    raise ValueError(f"Заполните параметры .env: {', '.join(missing_settings)}")

db_config = {
    'dbname': os.environ['POSTGRES_DB'],
    'user': os.environ['POSTGRES_USER'],
    'password': os.environ['POSTGRES_PASSWORD'],
    'host': os.environ['POSTGRES_HOST'],
    'port': int(os.environ['POSTGRES_PORT']),
    'connect_timeout': 10,
}
api_url = os.environ['PRODUCTS_API_URL'].strip()
PAGE_SIZE = 100

logger = logging.getLogger('dummyjson_etl')
logger.setLevel(logging.INFO)
logger.handlers.clear() # rerun cell without duplicate logs
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter('{asctime} | {levelname} | {message}', style='{'))
logger.addHandler(handler)
logger.propagate = False
pd.options.display.float_format = lambda value: f'{value:.2f}'

print('КОНФИГУРАЦИЯ ГОТОВА')


КОНФИГУРАЦИЯ ГОТОВА


## E - Extract

Для каждого запроса задаю таймаут. При временных сетевых ошибках и ответах 429/5xx делаю до трех повторов с exponential backoff о даа. Если total меняется между страницами или ответ неполный, останавливаю загрузку до обращения к БД

In [3]:
def extract_products(api_url, page_size=100):
    products = []
    total = None
    skip = 0
    retries = Retry(
        total=3,
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=['GET'],
        respect_retry_after_header=False,
    )
    with requests.Session() as session:
        session.mount('https://', HTTPAdapter(max_retries=retries))
        session.mount('http://', HTTPAdapter(max_retries=retries))
        while total is None or skip < total:
            try:
                response = session.get(api_url, params={'limit': page_size, 'skip': skip}, timeout=(5, 30))
                response.raise_for_status()
                otvet = response.json(parse_float=Decimal)
            except (requests.RequestException, ValueError) as error:
                logger.error(f'Не удалось получить страницу skip={skip}: {type(error).__name__}')
                raise RuntimeError('Ошибка API: проверьте доступность DummyJSON и повторите запуск') from None

            if not isinstance(otvet, dict):
                raise ValueError('API вернул не JSON-объект')
            page = otvet.get('products')
            page_total = otvet.get('total')
            if type(page_total) is not int or page_total <= 0:
                raise ValueError('API вернул некорректный total или пустой каталог')
            if total is None:
                total = page_total
            if page_total != total:
                raise ValueError('Каталог изменился во время выгрузки: запустите ноутбук заново')
            expected_size = min(page_size, total-skip)
            if not isinstance(page, list) or len(page) != expected_size or otvet.get('skip') != skip:
                raise ValueError(f'Неполная или неверная страница API: skip={skip}')
            products.extend(page)
            skip += len(page)
            logger.info(f'Extract: получено {len(products)} из {total} товаров')

    if len(products) != total:
        raise ValueError('Число извлеченных товаров не совпало с total')
    return products


raw_products = extract_products(api_url, PAGE_SIZE)
print(f'Товаров из API: {len(raw_products)}')


2026-11-22 02:20:12,434 | INFO | Extract: получено 100 из 194 товаров


2026-11-22 02:20:12,542 | INFO | Extract: получено 194 из 194 товаров


Товаров из API: 194


![ETL начинается в четверг, заканчивается в субботу](../memes_and_images/ETL_meme.jpeg)


## T - Transform

Обязательные поля проверяю до загрузки: уникальные положительные ID, непустые название и категория, корректные числовые значения и вложенные размеры. Цена, остаток и размеры не могут быть отрицательными, скидка должна быть от 0 до 100, рейтинг - от 0 до 5

Пустой или отсутствующий brand превращаю в NULL. Остальные обязательные значения не подменяю: поврежденный товар останавливает весь запуск. Поля, которые не нужны целевой таблице, не переношу

Расчетная цена после скидки - price * (1 - discountPercentage / 100), округление ROUND_HALF_UP

Еще добавляю положение рейтинга товара внутри его категории по всему текущему снимку. Сравниваю уже сохраняемые рейтинги Decimal с двумя знаками: одинаковый рейтинг получает одинаковые percentile и segment


### Исходный Price - цена до скидки

В [официальном примере корзины DummyJSON](https://dummyjson.com/docs/carts) price = 44.99, quantity = 4, total = 179.96, discountPercentage = 11.47 и discountedTotal = 159.32. Действительно, 44.99 * 4 = 179.96, а после скидки и округления получаем 159.32

Так как в таблице указана цена до скидки, и размер скидки - логично на стадии transform посчитать цену с учетом скидки


In [4]:
cart_price = Decimal('44.99')
cart_quantity = 4
cart_discount = Decimal('11.47')
cart_total = cart_price * cart_quantity
cart_discounted_total = (cart_total * (1-cart_discount/100)).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)

assert cart_total == Decimal('179.96')
assert cart_discounted_total == Decimal('159.32')
print(f'Проверка примера корзины: total={cart_total:.2f}, discountedTotal={cart_discounted_total:.2f}')


Проверка примера корзины: total=179.96, discountedTotal=159.32


### Рейтинг внутри категории

Для каждого товара считаю n - размер его категории, L - число товаров со строго меньшим рейтингом, E - число товаров с таким же рейтингом, включая сам товар. Перцентиль = 100 * (L + 0.5 * E) / n: половину группы с равным рейтингом помещаю ниже текущего положения

+ rating_percentile: перцентиль, сохраненный как Decimal с двумя знаками и округлением ROUND_HALF_UP

+ rating_segment: top20% при точном перцентиле >= 80, top50% при >= 50 и < 80, not ranked high при < 50

+ category_size: число всех товаров той же категории в текущем снимке

Если в категории один товар или все рейтинги одинаковы, записываю rating_percentile = NULL и rating_segment = not comparable


In [5]:
def pochistim_text(value, field, optional=False):
    if optional and value is None:
        return None
    if not isinstance(value, str):
        raise ValueError(f'{field}: ожидается строка')
    value = value.strip()
    if not value and not optional:
        raise ValueError(f'{field}: обязательная строка пуста')
    return value or None


def prochitaem_chislo(value, field, maximum):
    if isinstance(value, bool) or value is None or not isinstance(value, (int, float, Decimal, str)):
        raise ValueError(f'{field}: ожидается число')
    if isinstance(value, str) and not re.fullmatch(r'[+-]?(?:[0-9]+(?:\.[0-9]*)?|\.[0-9]+)(?:[eE][+-]?[0-9]+)?', value.strip()):
        raise ValueError(f'{field}: строка не содержит число')
    try:
        number = Decimal(str(value))
    except InvalidOperation:
        raise ValueError(f'{field}: некорректная запись числа') from None
    if not number.is_finite() or not Decimal('0') <= number <= maximum:
        raise ValueError(f'{field}: число вне допустимого диапазона')
    return number


def prochitaem_int(value, field, minimum):
    number = prochitaem_chislo(value, field, Decimal('2147483647'))
    if number != number.to_integral_value() or number < minimum:
        raise ValueError(f'{field}: ожидается целое число >= {minimum}')
    return int(number)


def enrich_rating_features(products_df):
    result = products_df.copy()
    rating_counts = (
        result.groupby(['category', 'rating'], sort=True)
        .size()
        .rename('equal_count')
        .reset_index()
    )
    categories = rating_counts.groupby('category', sort=False)
    rating_counts['category_size'] = categories['equal_count'].transform('sum')
    rating_counts['lower_count'] = categories['equal_count'].cumsum() - rating_counts['equal_count']
    rating_counts['distinct_count'] = categories['rating'].transform('size')

    features = {}
    for row in rating_counts.itertuples(index=False):
        numerator = 100 * row.lower_count + 50 * row.equal_count
        if row.category_size == 1 or row.distinct_count == 1:
            percentile = None
            segment = 'not comparable'
        else:
            # Сначала точный порог, потом округление. Float тут отдыхает
            if numerator >= 80 * row.category_size:
                segment = 'top20%'
            elif numerator >= 50 * row.category_size:
                segment = 'top50%'
            else:
                segment = 'not ranked high'
            percentile = (Decimal(int(numerator)) / Decimal(int(row.category_size))).quantize(
                Decimal('0.01'), rounding=ROUND_HALF_UP
            )
        features[(row.category, row.rating)] = (percentile, segment, int(row.category_size))

    rating_features = [features[(category, rating)] for category, rating in zip(result['category'], result['rating'])]
    result['rating_percentile'] = pd.Series([item[0] for item in rating_features], index=result.index, dtype=object)
    result['rating_segment'] = [item[1] for item in rating_features]
    result['category_size'] = [item[2] for item in rating_features]
    return result


def transform_products(products):
    if not products:
        raise ValueError('Пустой каталог: существующую таблицу не трогаем')
    rows = []
    seen_ids = set()
    kopeyka = Decimal('0.01') # название условное, валюты в ответе нет
    for product in products:
        if not isinstance(product, dict):
            raise ValueError('Товар должен быть JSON-объектом')
        product_id = prochitaem_int(product.get('id'), 'id', 1)
        if product_id in seen_ids:
            raise ValueError(f'Повторный id={product_id}: загрузка остановлена')
        seen_ids.add(product_id)
        price = prochitaem_chislo(product.get('price'), 'price', Decimal('9999999999.99'))
        discount = prochitaem_chislo(product.get('discountPercentage'), 'discountPercentage', Decimal('100'))
        rating = prochitaem_chislo(product.get('rating'), 'rating', Decimal('5'))
        dimensions = product.get('dimensions')
        if not isinstance(dimensions, dict):
            raise ValueError(f'id={product_id}: отсутствует объект dimensions')
        row = {
            'product_id': product_id,
            'title': pochistim_text(product.get('title'), 'title'),
            'category': pochistim_text(product.get('category'), 'category').lower(),
            'brand': pochistim_text(product.get('brand'), 'brand', optional=True),
            'price': price.quantize(kopeyka, rounding=ROUND_HALF_UP),
            'discount_percentage': discount.quantize(kopeyka, rounding=ROUND_HALF_UP),
            'discounted_price': (price * (1-discount/100)).quantize(kopeyka, rounding=ROUND_HALF_UP),
            'rating': rating.quantize(kopeyka, rounding=ROUND_HALF_UP),
            'stock': prochitaem_int(product.get('stock'), 'stock', 0),
        }
        for batman in ('width', 'height', 'depth'): # размеры тоже вытащим из JSON-пещеры
            dimension = prochitaem_chislo(dimensions.get(batman), batman, Decimal('99999999.99'))
            row[batman] = dimension.quantize(kopeyka, rounding=ROUND_HALF_UP)
        rows.append(row)

    result = enrich_rating_features(pd.DataFrame(rows))
    logger.info(f'Transform: проверено и подготовлено {len(result)} товаров')
    return result


products_df = transform_products(raw_products)
display(products_df.head(5))


2026-11-22 02:20:12,571 | INFO | Transform: проверено и подготовлено 194 товаров


,product_id,title,category,brand,price,discount_percentage,discounted_price,rating,stock,width,height,depth,rating_percentile,rating_segment,category_size
0,1,Essence Mascara Lash Princess,beauty,Essence,9.99,10.48,8.94,2.56,99,15.14,13.08,22.99,10.00,not ranked high,5
1,2,Eyeshadow Palette with Mirror,beauty,Glamour Beauty,19.99,18.19,16.35,2.86,34,9.26,22.47,27.67,30.00,not ranked high,5
2,3,Powder Canister,beauty,Velvet Touch,14.99,9.84,13.51,4.64,89,29.27,27.93,20.59,90.00,top20%,5
3,4,Red Lipstick,beauty,Chic Cosmetics,12.99,12.16,11.41,4.36,91,18.11,28.38,22.17,70.00,top50%,5
4,5,Red Nail Polish,beauty,Nail Couture,8.99,11.44,7.96,4.32,79,21.63,16.48,29.84,50.00,top50%,5


### Что получилось для beauty

Смотрю на все товары beauty, отсортированные по рейтингу: видны исходный рейтинг, положение внутри категории и итоговая группа. Размер категории остается общим для всех ее товаров


In [6]:
rating_columns = ['product_id', 'title', 'category', 'rating', 'rating_percentile', 'rating_segment', 'category_size']
rating_demo = products_df.loc[products_df['category'] == 'beauty', rating_columns]
display(rating_demo.sort_values(['rating', 'product_id']))


,product_id,title,category,rating,rating_percentile,rating_segment,category_size
0,1,Essence Mascara Lash Princess,beauty,2.56,10.00,not ranked high,5
1,2,Eyeshadow Palette with Mirror,beauty,2.86,30.00,not ranked high,5
4,5,Red Nail Polish,beauty,4.32,50.00,top50%,5
3,4,Red Lipstick,beauty,4.36,70.00,top50%,5
2,3,Powder Canister,beauty,4.64,90.00,top20%,5


### Распределение по группам

Показываю количество товаров каждого сегмента по всему снимку, включая отсутствующие сегменты с нулем. Доли зависят от размеров категорий и ничьих, поэтому ровно 20% товаров среди лидеров я не ожидаю


In [7]:
segment_order = ['top20%', 'top50%', 'not ranked high', 'not comparable']
segment_counts = products_df['rating_segment'].value_counts().reindex(segment_order, fill_value=0)
segment_distribution = segment_counts.rename_axis('rating_segment').reset_index(name='product_count')
segment_distribution['share_percent'] = segment_distribution['product_count'] / len(products_df) * 100
display(segment_distribution)


,rating_segment,product_count,share_percent
0,top20%,39,20.10
1,top50%,68,35.05
2,not ranked high,87,44.85
3,not comparable,0,0.00


![Что на самом деле делают дата-инженеры](../memes_and_images/ETL_meme_2.jpeg)

Бабушка жестко шарит


## L - Load

Очистка и вставка работают внутри одной транзакции. Никакого commit между ними: иначе при неудачной вставке мы остались бы с пустой таблицей, а это уже грустный ETL грустная песня

Проверка количества строк выполняется до commit. Контекст соединения делает rollback при исключении, внешний closing закрывает соединение и после успеха, и после ошибки смотря какой fabric смотря какой details

Перед очисткой применяю идемпотентную миграцию с новыми полями и ограничением для признаков рейтинга. Миграция, замена каталога и проверка ограничения находятся в одной транзакции: если что-то упало, откатываются и данные, и изменение схемы


In [8]:
def load_products(products_df, db_config):
    if products_df.empty:
        raise ValueError('Нельзя заменить каталог пустым набором')
    migration_sql = (env_path.parent / 'migrations' / '0001_rating_features.sql').read_text(encoding='utf-8')
    load_columns = [
        'product_id', 'title', 'category', 'brand', 'price', 'discount_percentage',
        'discounted_price', 'rating', 'stock', 'width', 'height', 'depth',
        'rating_percentile', 'rating_segment', 'category_size',
    ]
    records = list(products_df[load_columns].itertuples(index=False, name=None))
    insert_query = """
        INSERT INTO catalog.products (
            product_id, title, category, brand, price, discount_percentage,
            discounted_price, rating, stock, width, height, depth,
            rating_percentile, rating_segment, category_size
        ) VALUES %s
    """
    try:
        with closing(psycopg2.connect(**db_config)) as connection:
            with connection:
                with connection.cursor() as cursor:
                    cursor.execute(migration_sql)
                    cursor.execute('TRUNCATE TABLE catalog.products')
                    execute_values(cursor, insert_query, records, page_size=100)
                    cursor.execute('ALTER TABLE catalog.products VALIDATE CONSTRAINT products_rating_features_valid')
                    cursor.execute('SELECT COUNT(*) FROM catalog.products')
                    count = cursor.fetchone()[0]
                    if count != len(records):
                        raise ValueError('После вставки количество строк не совпало с выгрузкой')
    except psycopg2.Error as error:
        logger.error(f'Ошибка PostgreSQL: {type(error).__name__}, транзакция не сохранена')
        raise RuntimeError('Не удалось загрузить каталог: проверьте Docker, .env и init/init.sql') from None
    logger.info(f'Load: транзакция сохранена, строк в catalog.products: {count}')
    return count


loaded_count = load_products(products_df, db_config)
print('КАТАЛОГ В БАЗЕ, И ЭТО БАЗА')


2026-11-22 02:20:12,736 | INFO | Load: транзакция сохранена, строк в catalog.products: 194


КАТАЛОГ В БАЗЕ, И ЭТО БАЗА


## Проверка

Проверяю число строк и число уникальных ID отдельным соединением после commit. Первые пять товаров сортирую по ID, чтобы вывод был воспроизводимым. Для проверки идемпотентности запускаем ноутбук еще раз: при неизменном источнике количество строк остается тем же

Отдельным SQL-запросом заново вычисляю размер категории, число меньших и равных рейтингов через оконные функции. Сравниваю с сохраненными перцентилем и сегментом, проверяю диапазон и согласованность NULL с not comparable. Эта проверка не использует Python-функцию обогащения


In [9]:
try:
    with closing(psycopg2.connect(**db_config)) as connection:
        with connection:
            with connection.cursor() as cursor:
                cursor.execute('SELECT COUNT(*), COUNT(DISTINCT product_id) FROM catalog.products')
                row_count, unique_count = cursor.fetchone()
                cursor.execute('SELECT * FROM catalog.products ORDER BY product_id LIMIT 5')
                first_rows = cursor.fetchall()
                column_names = [column.name for column in cursor.description]
                cursor.execute('''
                    WITH ranked AS (
                        SELECT
                            product_id, category_size, rating_percentile, rating_segment,
                            COUNT(*) OVER (PARTITION BY category) AS expected_size,
                            RANK() OVER (PARTITION BY category ORDER BY rating) - 1 AS lower_count,
                            COUNT(*) OVER (PARTITION BY category, rating) AS equal_count,
                            MIN(rating) OVER (PARTITION BY category) AS minimum_rating,
                            MAX(rating) OVER (PARTITION BY category) AS maximum_rating
                        FROM catalog.products
                    ), expected AS (
                        SELECT *,
                            CASE
                                WHEN expected_size = 1 OR minimum_rating = maximum_rating THEN NULL
                                ELSE (100 * lower_count + 50 * equal_count)::NUMERIC / expected_size
                            END AS exact_percentile
                        FROM ranked
                    )
                    SELECT COUNT(*) FROM expected
                    WHERE category_size IS DISTINCT FROM expected_size
                        OR rating_percentile IS DISTINCT FROM ROUND(exact_percentile, 2)
                        OR rating_segment IS DISTINCT FROM CASE
                            WHEN exact_percentile IS NULL THEN 'not comparable'
                            WHEN exact_percentile >= 80 THEN 'top20%'
                            WHEN exact_percentile >= 50 THEN 'top50%'
                            ELSE 'not ranked high'
                        END
                        OR rating_percentile NOT BETWEEN 0 AND 100
                        OR (rating_percentile IS NULL) IS DISTINCT FROM (rating_segment = 'not comparable')
                ''')
                rating_errors = cursor.fetchone()[0]
except psycopg2.Error as error:
    logger.error(f'Не удалось проверить PostgreSQL: {type(error).__name__}')
    raise RuntimeError('Проверка БД не завершена: проверьте соединение и повторите запуск') from None

if row_count != loaded_count or unique_count != loaded_count:
    raise ValueError('Проверка не пройдена: число строк или уникальных ID не совпало')

if rating_errors != 0:
    raise ValueError(f'Проверка признаков рейтинга не пройдена: ошибок {rating_errors}')

print(f'Строк в catalog.products: {row_count}')
print(f'Уникальных product_id: {unique_count}')
print(f'Товаров извлечено из API: {len(raw_products)}')
print(f'Ошибок в признаках рейтинга: {rating_errors}')
display(pd.DataFrame(first_rows, columns=column_names))


Строк в catalog.products: 194
Уникальных product_id: 194
Товаров извлечено из API: 194
Ошибок в признаках рейтинга: 0


,product_id,title,category,brand,price,discount_percentage,discounted_price,rating,stock,width,height,depth,rating_percentile,rating_segment,category_size
0,1,Essence Mascara Lash Princess,beauty,Essence,9.99,10.48,8.94,2.56,99,15.14,13.08,22.99,10.00,not ranked high,5
1,2,Eyeshadow Palette with Mirror,beauty,Glamour Beauty,19.99,18.19,16.35,2.86,34,9.26,22.47,27.67,30.00,not ranked high,5
2,3,Powder Canister,beauty,Velvet Touch,14.99,9.84,13.51,4.64,89,29.27,27.93,20.59,90.00,top20%,5
3,4,Red Lipstick,beauty,Chic Cosmetics,12.99,12.16,11.41,4.36,91,18.11,28.38,22.17,70.00,top50%,5
4,5,Red Nail Polish,beauty,Nail Couture,8.99,11.44,7.96,4.32,79,21.63,16.48,29.84,50.00,top50%,5


## Что получили

Мы загрузили полный каталог, проверили обязательные значения и сохранили его одним снимком. Фактическое количество товаров и первые строки показаны выше. Теперь я официально могу сказать, что мы все сделали мы молодцы)
